# 파생변수 생성

담당: 황유진 관련이슈: #39

## 목적
기존 EDA 결과를 종합하여 연속소둔로의 데이터 구조와 조업조건을 이해하고, 정상적인 조업 변화와 설비의 비정상적인 변화를 구분하는 데 활용할 수 있는 모델 입력 파생지표 후보를 제안한다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


In [ ]:
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])

df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

df.head()

In [ ]:
print("데이터 크기:", df.shape)
print("LOT 개수:", df["LOT_NO"].nunique())
print("측정 시작:", df["MEAS_DT"].min())
print("측정 종료:", df["MEAS_DT"].max())

In [ ]:
tc_cols = [col for col in df.columns if col.startswith("TC-")]
op_cols = [col for col in df.columns if col.startswith("OP-")]

print("TC 컬럼:", tc_cols)
print("OP 컬럼:", op_cols)
print("TC 개수:", len(tc_cols))
print("OP 개수:", len(op_cols))

# TC/OP/CP 자체에서 만드는 시계열 파생변수
- "지금 센서값이 평소와 다르게 움직이고 있나?" 를 보는 변수

## 1. 직전 시점 대비 변화량 (diff)

의미: 현재값 - 바로 직전 시점의 값

직전 값보다 얼마나 변했는지를 보기 위함. 이상 구간에서는 온도나 제어출력이 갑자기 튀는 경우가 있을 수 있음.

In [ ]:
df_feature = df.copy()

for col in tc_cols + op_cols:
    df_feature[f"{col}_diff"] = (
        df_feature.groupby("LOT_NO")[col].diff()
    )

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "TC-Z3", "TC-Z3_diff"]
].head(10)

## 절대 변화량 (abs_diff)

의미: 직전 시점과 비교해서 얼마나 많이 변했는지 크기만 보는 값

상승/하락 상관없이 갑자기 크게 변했는지를 보기 위함

In [ ]:
for col in tc_cols + op_cols:
    df_feature[f"{col}_abs_diff"] = (
        df_feature[f"{col}_diff"].abs()
    )

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "TC-Z3", "TC-Z3_diff", "TC-Z3_abs_diff"]
].head(10)

## 3. 이동평균 (roll_mean_60)

의미: 현재 시점을 포함해서 최근 여러 관측값의 평균

현재 LOT의 최근 정상 기준선

- 주의: 60개 관측값은 정확한 60초와 동일하지 않으며, N = 60은 향후 검증이 필요한 후보값이다

In [ ]:
for col in tc_cols + op_cols:
    df_feature[f"{col}_roll_mean_60"] = (
        df_feature.groupby("LOT_NO")[col]
        .transform(lambda x: x.shift(1).rolling(window=60, min_periods=1).mean())
    )

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "TC-Z3", "TC-Z3_roll_mean_60"]
].head(70)

## 4. 이동표준편차 (roll_std_60)

의미: 최근 여러 관측값이 얼마나 흔들리고 있는지를 보는 값

불안정·진동 같은 변동성 탐지

- 주의: window=60 은 확정된 최적값이 아니라 후보값이며 모델 검증 단계에서 조정한다

In [ ]:
for col in tc_cols + op_cols:
    df_feature[f"{col}_roll_std_60"] = (
        df_feature.groupby("LOT_NO")[col]
        .transform(lambda x: x.rolling(window=60, min_periods=2).std())
    )

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "TC-Z3", "TC-Z3_roll_std_60"]
].head(70)

## 5. 이동평균 대비 편차 (deviation)

의미: 현재값 - 최근 이동평균

지금 값이 최근 기준 수준에서 얼마나 벗어났는지

In [ ]:
for col in tc_cols + op_cols:
    df_feature[f"{col}_deviation"] = (
        df_feature[col] - df_feature[f"{col}_roll_mean_60"]
    )

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "TC-Z3", "TC-Z3_roll_mean_60", "TC-Z3_deviation"]
].head(70)

## 6. 이동 표준점수 (roll_zscore)

의미: 현재값이 최근 평균에서 최근 변동폭 기준으로 얼마나 멀리 떨어져 있는지 보는 값

In [ ]:
for col in tc_cols + op_cols:
    df_feature[f"{col}_roll_zscore"] = (
        df_feature[f"{col}_deviation"]
        / df_feature[f"{col}_roll_std_60"]
    )

In [ ]:
df_feature[
    [
        "LOT_NO",
        "MEAS_DT",
        "TC-Z3",
        "TC-Z3_roll_mean_60",
        "TC-Z3_roll_std_60",
        "TC-Z3_roll_zscore"
    ]
].head(70)

# 정상적인 조업조건을 알려주는 파생변수
- "지금은 강종이 바뀌어서 정상 기준도 바뀔 수 있는 상황이야" 라는 정보를 주는 변수

## 1. LOT 시작 후 경과시간 (LOT_ELAPSED_SEC)

의미: 현재 행이 해당 LOT 생산을 시작한 뒤 몇 초가 지난 시점인지

LOT 초반 과도상태와 안정구간 구분

In [ ]:
lot_start = df_feature.groupby("LOT_NO")["MEAS_DT"].transform("min")

df_feature["LOT_ELAPSED_SEC"] = (
    df_feature["MEAS_DT"] - lot_start
).dt.total_seconds()

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "LOT_ELAPSED_SEC"]
].head(10)

## 2. LOT별  현재 강종 (STEEL_GRADE)

의미: 현재 생산 강종

강종 때문에 생기는 정상 baseline 변화 설명

In [ ]:
event = pd.read_csv(
    "../data/G-02_조업이벤트.csv",
    encoding="utf-8-sig"
)

event["EVT_DT"] = pd.to_datetime(event["EVT_DT"])

grade_event = event[
    (event["PLANT_CD"] == "CR1")
    & (event["EVT_TYPE"] == "강종변경")
].copy()

In [ ]:
lot_grade = (
    df_feature.groupby("LOT_NO", as_index=False)["MEAS_DT"]
    .min()
    .rename(columns={"MEAS_DT": "LOT_START_DT"})
)

In [ ]:
lot_grade = lot_grade.merge(
    grade_event[["EVT_DT", "BEF_VAL", "AFT_VAL"]],
    left_on="LOT_START_DT",
    right_on="EVT_DT",
    how="left"
)

lot_grade = lot_grade.rename(
    columns={
        "BEF_VAL": "PREV_STEEL_GRADE",
        "AFT_VAL": "STEEL_GRADE"
    }
)

In [ ]:
lot_grade[
    ["LOT_NO", "LOT_START_DT", "PREV_STEEL_GRADE", "STEEL_GRADE"]
]

In [ ]:
df_feature = df_feature.merge(
    lot_grade[["LOT_NO", "STEEL_GRADE"]],
    on="LOT_NO",
    how="left"
)

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "STEEL_GRADE", "TC-Z3", "OP-Z3"]
].head(10)

# 보조변수

## 3. 측정 간격 (DT_SEC)

의미: 같은 LOT 안에서 직전 측정값과 현재 측정값 사이가 몇 초인지

불규칙 측정간격 확인

In [ ]:
df_feature["DT_SEC"] = (
    df_feature.groupby("LOT_NO")["MEAS_DT"]
    .diff()
    .dt.total_seconds()
)

In [ ]:
df_feature[
    ["LOT_NO", "MEAS_DT", "DT_SEC"]
].head(10)

In [ ]:
print(df_feature["DT_SEC"].value_counts().sort_index())

## 확인 결과

| 제안 변수 | 의미 |
|---|---|
| `diff` | 직전 값 대비 변화량 |
| `abs_diff` | 변화량의 절대 크기 |
| `roll_mean_60` | 최근 60개 관측값의 평균 |
| `roll_std_60` | 최근 60개 관측값의 변동성 |
| `deviation` | 현재값과 최근 평균의 차이 |
| `roll_zscore` | 최근 변동성을 고려한 상대적 이탈 정도 |
| `LOT_ELAPSED_SEC` | LOT 시작 후 경과시간 |
| `STEEL_GRADE` | 강종에 따른 조업조건 반영 |
| `DT_SEC` | 측정 간격 확인용 보조 변수 |

기존 EDA 결과를 바탕으로 센서의 변화, 변동성, 기준선 이탈,
강종 및 LOT 진행상태를 표현할 수 있는 변수들을 모델 입력 후보로 제안하였다.

※ `roll_mean_60`, `roll_std_60`의 60은 정확한 60초가 아니라 최근 60개 관측값을 의미한다.